<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/4typowanie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_typowanie.py

from __future__ import annotations

import json
from pathlib import Path
from typing import (
    Any,
    Callable,
    Final,
    Literal,
    NoReturn,
    Optional,
    Self,
    TypeAlias,
    TypedDict,
    Union,
)


FOLDER_PROJEKTU: Final[Path] = Path(
    "/content/drive/MyDrive/projekt_test"
)

WERSJA_MODULU: Final[str] = "1.0"


Ticker: TypeAlias = str
Cena: TypeAlias = float
Wolumen: TypeAlias = int
Procent: TypeAlias = float

Liczba: TypeAlias = Union[int, float]

StatusLiteral: TypeAlias = Literal[
    "passed",
    "failed",
    "not_evaluated",
    "error",
]

Kierunek: TypeAlias = Literal[
    "dodatni",
    "ujemny",
    "neutralny",
]

FiltrBool: TypeAlias = Callable[
    [["DaneSpolki"]],
    bool,
]

StrategiaPunktowa: TypeAlias = Callable[
    [["DaneSpolki"]],
    float,
]


class SpolkaTD(TypedDict):
    ticker: str
    nazwa: str
    sektor: str
    branza: Optional[str]
    kraj: Optional[str]
    tagi: list[str]
    rynek: str


class StatystykiTD(TypedDict):
    liczba_notowan: int
    srednia_cena: float
    momentum: float
    zmiennosc: float
    ostatnia_cena: float


class NotowanieTD(TypedDict):
    data: str
    open: float
    high: float
    low: float
    close: float
    volume: int


class WarunkiTD(TypedDict):
    wartosc_flag: int
    spelnione: list[str]


class DaneEnumTD(TypedDict):
    spolka: SpolkaTD
    typy_filtrow: list[str]
    status: str
    ranking: int
    warunki: WarunkiTD
    statystyki: StatystykiTD
    notowania: list[NotowanieTD]


class TypowanieTD(TypedDict):
    wersja: str
    ticker: str
    status: StatusLiteral
    kierunek_momentum: Kierunek
    spolka: SpolkaTD
    statystyki: StatystykiTD
    notowania: list[NotowanieTD]
    typy_filtrow: list[str]
    ranking: int
    warunki: WarunkiTD


class DaneSpolki:
    ticker: Ticker
    srednia_cena: Cena
    momentum: Procent
    zmiennosc: Procent
    ostatnia_cena: Cena
    liczba_notowan: int
    ranking: int
    status: StatusLiteral
    tagi: list[str]

    def __init__(
        self,
        ticker: Ticker,
        srednia_cena: Cena,
        momentum: Procent,
        zmiennosc: Procent,
        ostatnia_cena: Cena,
        liczba_notowan: int,
        ranking: int,
        status: StatusLiteral,
        tagi: Optional[list[str]] = None,
    ) -> None:

        self.ticker = ticker
        self.srednia_cena = srednia_cena
        self.momentum = momentum
        self.zmiennosc = zmiennosc
        self.ostatnia_cena = ostatnia_cena
        self.liczba_notowan = liczba_notowan
        self.ranking = ranking
        self.status = status

        self.tagi = (
            list(tagi)
            if tagi is not None
            else []
        )

    @classmethod
    def z_danych_enum(
        cls,
        dane: DaneEnumTD,
    ) -> Self:

        spolka: SpolkaTD = dane["spolka"]

        statystyki: StatystykiTD = (
            dane["statystyki"]
        )

        status: StatusLiteral = (
            waliduj_status(
                dane["status"]
            )
        )

        return cls(
            ticker=str(
                spolka["ticker"]
            ),
            srednia_cena=float(
                statystyki["srednia_cena"]
            ),
            momentum=float(
                statystyki["momentum"]
            ),
            zmiennosc=float(
                statystyki["zmiennosc"]
            ),
            ostatnia_cena=float(
                statystyki["ostatnia_cena"]
            ),
            liczba_notowan=int(
                statystyki["liczba_notowan"]
            ),
            ranking=int(
                dane["ranking"]
            ),
            status=status,
            tagi=list(
                spolka["tagi"]
            ),
        )

    def dodaj_tag(
        self,
        tag: str,
    ) -> Self:

        tag = tag.strip().lower()

        if tag and tag not in self.tagi:
            self.tagi.append(
                tag
            )

        return self


def blad_typowania(
    komunikat: str,
) -> NoReturn:

    raise TypeError(
        komunikat
    )


def waliduj_status(
    status: str,
) -> StatusLiteral:

    dozwolone: tuple[
        StatusLiteral,
        ...
    ] = (
        "passed",
        "failed",
        "not_evaluated",
        "error",
    )

    if status not in dozwolone:
        blad_typowania(
            f"niepoprawny status: {status}"
        )

    return status  # type: ignore[return-value]


def pobierz_float(
    wartosc: Any,
    nazwa: str,
) -> float:

    if isinstance(
        wartosc,
        bool,
    ):
        blad_typowania(
            f"{nazwa} nie moze byc bool"
        )

    try:
        return float(
            wartosc
        )

    except (
        TypeError,
        ValueError,
    ) as e:

        raise TypeError(
            f"{nazwa} musi byc liczba"
        ) from e


def pobierz_int(
    wartosc: Any,
    nazwa: str,
) -> int:

    if isinstance(
        wartosc,
        bool,
    ):
        blad_typowania(
            f"{nazwa} nie moze byc bool"
        )

    try:
        return int(
            wartosc
        )

    except (
        TypeError,
        ValueError,
    ) as e:

        raise TypeError(
            f"{nazwa} musi byc int"
        ) from e


def okresl_kierunek(
    momentum: float,
) -> Kierunek:

    if momentum > 0:
        return "dodatni"

    if momentum < 0:
        return "ujemny"

    return "neutralny"


def filtr_dodatnie_momentum(
    spolka: DaneSpolki,
) -> bool:

    return (
        spolka.momentum > 0
    )


def filtr_niska_zmiennosc(
    spolka: DaneSpolki,
) -> bool:

    return (
        spolka.zmiennosc < 2.0
    )


def strategia_momentum(
    spolka: DaneSpolki,
) -> float:

    return float(
        spolka.momentum
    )


def wykonaj_filtr(
    spolka: DaneSpolki,
    filtr: FiltrBool,
) -> bool:

    return filtr(
        spolka
    )


def wykonaj_strategie(
    spolka: DaneSpolki,
    strategia: StrategiaPunktowa,
) -> float:

    return strategia(
        spolka
    )


def pokaz_annotations(
    klasa: type[Any],
) -> None:

    print(
        f"\nTYPE ANNOTATIONS: "
        f"{klasa.__name__}"
    )

    annotations: dict[
        str,
        Any
    ] = dict(
        getattr(
            klasa,
            "__annotations__",
            {},
        )
    )

    for nazwa, typ in (
        annotations.items()
    ):
        print(
            nazwa,
            "->",
            typ,
        )


def waliduj_runtime(
    obiekt: DaneSpolki,
) -> None:

    annotations: dict[
        str,
        Any
    ] = dict(
        obiekt.__class__.__annotations__
    )

    for nazwa in annotations:

        if not hasattr(
            obiekt,
            nazwa,
        ):
            blad_typowania(
                f"brak pola: {nazwa}"
            )

        wartosc: Any = getattr(
            obiekt,
            nazwa,
        )

        if wartosc is None:
            blad_typowania(
                f"pole {nazwa} "
                "nie moze byc None"
            )


def wczytaj_dane_z_modulu_3(
    ticker: Ticker,
    folder: Path = FOLDER_PROJEKTU,
) -> DaneEnumTD:

    ticker = (
        ticker
        .strip()
        .upper()
    )

    plik: Path = (
        folder
        / f"{ticker}_enum.json"
    )

    if not plik.exists():
        raise FileNotFoundError(
            f"brak pliku z modulu 3: "
            f"{plik}"
        )

    with open(
        plik,
        "r",
        encoding="utf-8",
    ) as f:

        surowe: Any = json.load(
            f
        )

    if not isinstance(
        surowe,
        dict,
    ):
        blad_typowania(
            "glowna struktura JSON "
            "musi byc slownikiem"
        )

    wymagane: tuple[
        str,
        ...
    ] = (
        "spolka",
        "typy_filtrow",
        "status",
        "ranking",
        "warunki",
        "statystyki",
        "notowania",
    )

    for klucz in wymagane:

        if klucz not in surowe:
            blad_typowania(
                f"brak pola: {klucz}"
            )

    spolka_raw: Any = (
        surowe["spolka"]
    )

    stat_raw: Any = (
        surowe["statystyki"]
    )

    warunki_raw: Any = (
        surowe["warunki"]
    )

    notowania_raw: Any = (
        surowe["notowania"]
    )

    if not isinstance(
        spolka_raw,
        dict,
    ):
        blad_typowania(
            "spolka musi byc dict"
        )

    if not isinstance(
        stat_raw,
        dict,
    ):
        blad_typowania(
            "statystyki musza byc dict"
        )

    if not isinstance(
        warunki_raw,
        dict,
    ):
        blad_typowania(
            "warunki musza byc dict"
        )

    if not isinstance(
        notowania_raw,
        list,
    ):
        blad_typowania(
            "notowania musza byc lista"
        )

    spolka: SpolkaTD = {
        "ticker":
            str(
                spolka_raw.get(
                    "ticker",
                    ticker,
                )
            ),

        "nazwa":
            str(
                spolka_raw.get(
                    "nazwa",
                    "",
                )
            ),

        "sektor":
            str(
                spolka_raw.get(
                    "sektor",
                    "nieznany",
                )
            ),

        "branza":
            (
                None
                if spolka_raw.get(
                    "branza"
                ) is None
                else str(
                    spolka_raw.get(
                        "branza"
                    )
                )
            ),

        "kraj":
            (
                None
                if spolka_raw.get(
                    "kraj"
                ) is None
                else str(
                    spolka_raw.get(
                        "kraj"
                    )
                )
            ),

        "tagi":
            [
                str(x)
                for x in spolka_raw.get(
                    "tagi",
                    [],
                )
            ],

        "rynek":
            str(
                spolka_raw.get(
                    "rynek",
                    "nieznany",
                )
            ),
    }

    statystyki: StatystykiTD = {
        "liczba_notowan":
            pobierz_int(
                stat_raw.get(
                    "liczba_notowan"
                ),
                "liczba_notowan",
            ),

        "srednia_cena":
            pobierz_float(
                stat_raw.get(
                    "srednia_cena"
                ),
                "srednia_cena",
            ),

        "momentum":
            pobierz_float(
                stat_raw.get(
                    "momentum"
                ),
                "momentum",
            ),

        "zmiennosc":
            pobierz_float(
                stat_raw.get(
                    "zmiennosc"
                ),
                "zmiennosc",
            ),

        "ostatnia_cena":
            pobierz_float(
                stat_raw.get(
                    "ostatnia_cena"
                ),
                "ostatnia_cena",
            ),
    }

    warunki: WarunkiTD = {
        "wartosc_flag":
            pobierz_int(
                warunki_raw.get(
                    "wartosc_flag",
                    0,
                ),
                "wartosc_flag",
            ),

        "spelnione":
            [
                str(x)
                for x
                in warunki_raw.get(
                    "spelnione",
                    [],
                )
            ],
    }

    notowania: list[
        NotowanieTD
    ] = []

    for rekord in notowania_raw:

        if not isinstance(
            rekord,
            dict,
        ):
            blad_typowania(
                "rekord notowania "
                "musi byc dict"
            )

        notowanie: NotowanieTD = {
            "data":
                str(
                    rekord.get(
                        "data",
                        "",
                    )
                ),

            "open":
                pobierz_float(
                    rekord.get(
                        "open"
                    ),
                    "open",
                ),

            "high":
                pobierz_float(
                    rekord.get(
                        "high"
                    ),
                    "high",
                ),

            "low":
                pobierz_float(
                    rekord.get(
                        "low"
                    ),
                    "low",
                ),

            "close":
                pobierz_float(
                    rekord.get(
                        "close"
                    ),
                    "close",
                ),

            "volume":
                pobierz_int(
                    rekord.get(
                        "volume"
                    ),
                    "volume",
                ),
        }

        notowania.append(
            notowanie
        )

    status: StatusLiteral = (
        waliduj_status(
            str(
                surowe["status"]
            )
        )
    )

    wynik: DaneEnumTD = {
        "spolka":
            spolka,

        "typy_filtrow":
            [
                str(x)
                for x
                in surowe[
                    "typy_filtrow"
                ]
            ],

        "status":
            status,

        "ranking":
            pobierz_int(
                surowe["ranking"],
                "ranking",
            ),

        "warunki":
            warunki,

        "statystyki":
            statystyki,

        "notowania":
            notowania,
    }

    return wynik


def przygotuj_dane_typowane(
    dane: DaneEnumTD,
) -> TypowanieTD:

    obiekt: DaneSpolki = (
        DaneSpolki.z_danych_enum(
            dane
        )
    )

    waliduj_runtime(
        obiekt
    )

    kierunek: Kierunek = (
        okresl_kierunek(
            obiekt.momentum
        )
    )

    filtr_momentum: FiltrBool = (
        filtr_dodatnie_momentum
    )

    filtr_zmiennosc: FiltrBool = (
        filtr_niska_zmiennosc
    )

    strategia: StrategiaPunktowa = (
        strategia_momentum
    )

    momentum_ok: bool = (
        wykonaj_filtr(
            obiekt,
            filtr_momentum,
        )
    )

    zmiennosc_ok: bool = (
        wykonaj_filtr(
            obiekt,
            filtr_zmiennosc,
        )
    )

    wynik_strategii: float = (
        wykonaj_strategie(
            obiekt,
            strategia,
        )
    )

    obiekt.dodaj_tag(
        f"momentum_ok_{momentum_ok}"
    )

    obiekt.dodaj_tag(
        f"zmiennosc_ok_{zmiennosc_ok}"
    )

    obiekt.dodaj_tag(
        f"score_{round(wynik_strategii, 2)}"
    )

    spolka: SpolkaTD = dict(
        dane["spolka"]
    )  # type: ignore[assignment]

    spolka["tagi"] = list(
        obiekt.tagi
    )

    wynik: TypowanieTD = {
        "wersja":
            WERSJA_MODULU,

        "ticker":
            obiekt.ticker,

        "status":
            obiekt.status,

        "kierunek_momentum":
            kierunek,

        "spolka":
            spolka,

        "statystyki":
            dane["statystyki"],

        "notowania":
            dane["notowania"],

        "typy_filtrow":
            dane["typy_filtrow"],

        "ranking":
            dane["ranking"],

        "warunki":
            dane["warunki"],
    }

    return wynik


def zapisz_dane_typowane(
    ticker: Ticker,
    dane: TypowanieTD,
    folder: Path = FOLDER_PROJEKTU,
) -> Path:

    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    ticker = (
        ticker
        .strip()
        .upper()
    )

    plik: Path = (
        folder
        / f"{ticker}_typed.json"
    )

    with open(
        plik,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            dane,
            f,
            ensure_ascii=False,
            indent=2,
        )

    return plik


def run() -> None:

    ticker: Ticker = input(
        "podaj ticker: "
    ).strip().upper()

    print(
        "\nwczytywanie danych "
        "zapisanych przez modul 3..."
    )

    dane: DaneEnumTD = (
        wczytaj_dane_z_modulu_3(
            ticker
        )
    )

    print(
        "wczytano:",
        ticker
    )

    dane_typowane: TypowanieTD = (
        przygotuj_dane_typowane(
            dane
        )
    )

    plik: Path = (
        zapisz_dane_typowane(
            ticker,
            dane_typowane,
        )
    )

    print(
        "\nSTATUS:",
        dane_typowane["status"]
    )

    print(
        "KIERUNEK MOMENTUM:",
        dane_typowane[
            "kierunek_momentum"
        ]
    )

    print(
        "LICZBA NOTOWAN:",
        dane_typowane[
            "statystyki"
        ]["liczba_notowan"]
    )

    print(
        "MOMENTUM:",
        round(
            dane_typowane[
                "statystyki"
            ]["momentum"],
            2,
        ),
        "%",
    )

    print(
        "ZMIENNOSC:",
        round(
            dane_typowane[
                "statystyki"
            ]["zmiennosc"],
            2,
        ),
        "%",
    )

    pokaz_annotations(
        DaneSpolki
    )

    print(
        "\nzapisano dane dla "
        "kolejnego modulu:"
    )

    print(
        plik
    )

    print(
        "\nMODUL TYPOWANIA "
        "DZIALA POPRAWNIE"
    )